# Gold dimension -- `dbo.dim_date`

Calendar, widened to whole calendar years so Power BI time intelligence has a contiguous full-year table to work against. Days generated as padding carry _is_generated = true and null attributes -- they exist so DATEADD and SAMEPERIODLASTYEAR resolve, not to be counted.

**Source:** `silver.stg_calendar_date`  
**SCD:** `type1`  
**Surrogate key:** `date_sk`  
**Tracked columns:** `n/a`

> GENERATED FILE -- DO NOT EDIT.
Produced by framework/generators/generate_notebooks.py from the project spec set. Edit the spec and regenerate; hand edits are overwritten and will fail the notebook-lint gate.


In [ ]:
# Parameters -- overridden per environment by the deployment pipeline.
# See 05-deployment.yaml `parameterisation`.
# Reads from lh_silver, writes to wh_gold. Both must be
# attached to this notebook; wh_gold must be the DEFAULT so an
# unqualified write cannot land in the wrong item.
target_item = "wh_gold"
source_item = "lh_silver"
environment = "dev"
dq_failure_action = "warn"

import sys
from datetime import datetime

from pyspark.sql import functions as F

from ttfabric.cleansing import RuleContext, get_rule
from ttfabric.quality import DQRunLog

load_id = f"load_{datetime.utcnow():%Y%m%d_%H%M%S}"

def resolve_table(name: str):
    """Resolve a spec table reference to a DataFrame.

    Deliberately UNQUALIFIED, so the read lands in the default lakehouse.

    Rules reference tables in their OWN layer -- enforce_referential_integrity
    against dim_products, recompute_total_from_lines against fct_order_items --
    and those peers live in the item this notebook writes to, not the one it
    reads its source from. Qualifying with source_item sent them to
    lh_bronze.dim_products, which does not and should not exist.

    The single cross-item read, this table's own bronze source, is qualified
    explicitly at the call site instead.
    """
    bare = name.split(".")[-1]
    return spark.read.table(bare)

ctx = RuleContext(
    spark=spark,
    load_id=load_id,
    environment=environment,
    table="dim_date",
    resolve_table=resolve_table,
    apply_masking=(environment in ("uat", "prod")),
)

dq = DQRunLog(spark, load_id=load_id, layer="gold", table_name="dim_date")
print(f"load_id={load_id}  environment={environment}  table=dim_date")

from ttfabric.warehouse import gold_target

gold = gold_target(
    spark,
    warehouse="wh_gold",
    schema="dbo",
    write_mode="warehouse_connector",
)


In [ ]:
# ---- Read silver -------------------------------------------------
src = spark.read.table(f"{source_item}.stg_calendar_date")
print(f"read {src.count():,} rows from stg_calendar_date")


In [ ]:
# ---- Project to the target schema --------------------------------
# Renames come from mappings/gold.yaml `columns`. Applied before the
# business rules, which are written against target names.
src = src.select(
    F.col("date_key"),
    F.col("calendar_date").alias("full_date"),
    F.col("is_work_day"),
    F.col("day_is_holiday"),
    F.col("day_name"),
    F.col("date_range"),
)


In [ ]:
# Business rule: year_number
# 
src = src.withColumn("year_number", F.expr("""year(full_date)"""))


In [ ]:
# Business rule: quarter_number
# 
src = src.withColumn("quarter_number", F.expr("""quarter(full_date)"""))


In [ ]:
# Business rule: month_number
# 
src = src.withColumn("month_number", F.expr("""month(full_date)"""))


In [ ]:
# Business rule: month_name
# 
src = src.withColumn("month_name", F.expr("""date_format(full_date, 'MMMM')"""))


In [ ]:
# Business rule: year_month
# 
src = src.withColumn("year_month", F.expr("""year(full_date) * 100 + month(full_date)"""))


In [ ]:
# Business rule: day_of_week
# 
src = src.withColumn("day_of_week", F.expr("""dayofweek(full_date)"""))


In [ ]:
# ---- SCD type 1 overwrite ----------------------------------------
from ttfabric.dimensions import assign_surrogate_key

out = assign_surrogate_key(src, "date_sk", ['date_key'])
gold.write(out, "dim_date")


In [ ]:
# ---- Unknown member ----------------------------------------------
# Guarantees an unmatched fact still joins rather than vanishing
# from a report without trace.
from ttfabric.dimensions import ensure_unknown_member

ensure_unknown_member(
    spark,
    table="dim_date",
    surrogate_key="date_sk",
    key_value=-1,
    defaults={'date_key': -1, 'calendar_date': '1900-01-01', 'is_work_day': 0, 'day_is_holiday': 0, 'day_name': 'Unknown', 'date_range': 'Unknown'},
    gold=gold,
)
dq.flush()
